# DailyETLMain — Databricks Migration

Generated from SSIS package by the migration agent.

| Field | Value |
| --- | --- |
| Steps | 13 |
| Unresolved | 0 |
| Planner | deterministic |


In [0]:
# Databricks Job parameters
# When run via a Databricks Job, each task passes
# step_id to execute only its portion.
# When run interactively, step_id="all" runs
# every step in dependency order.
dbutils.widgets.text('step_id', 'all', 'Step ID to execute')
dbutils.widgets.text('catalog', 'main', 'Target catalog')
dbutils.widgets.text('schema', 'default', 'Target schema')
_step_id = dbutils.widgets.get('step_id')
_catalog = dbutils.widgets.get('catalog')
_schema = dbutils.widgets.get('schema')


# Databricks Job parameters
# When run via a Databricks Job, each task passes
# step_id to execute only its portion.
# When run interactively, step_id="all" runs
# every step in dependency order.
dbutils.widgets.text('step_id', 'all', 'Step ID to execute')
dbutils.widgets.text('catalog', 'main', 'Target catalog')
dbutils.widgets.text('schema', 'default', 'Target schema')
dbutils.widgets.text('new_cutoff', '', 'New ETL cutoff timestamp (leave empty for current time)')

_step_id = dbutils.widgets.get('step_id')
_catalog = dbutils.widgets.get('catalog')
_schema = dbutils.widgets.get('schema')

# Set the active catalog so two-part table names resolve correctly
# e.g. Integration.City_Staging -> <catalog>.Integration.City_Staging
spark.sql(f'USE CATALOG `{_catalog}`')

# Set new_cutoff to user-provided value or current timestamp
_new_cutoff_raw = dbutils.widgets.get('new_cutoff')
if _new_cutoff_raw:
    new_cutoff = _new_cutoff_raw
else:
    new_cutoff = spark.sql('SELECT current_timestamp() AS ts').collect()[0]['ts'].strftime('%Y-%m-%d %H:%M:%S')

print(f'Step ID: {_step_id}')
print(f'Catalog: {_catalog}')
print(f'New cutoff: {new_cutoff}')


In [ ]:
# ============================================================
# ETL Helper Functions
# ============================================================
# These reproduce the control-flow logic from the original
# SSIS package (GetLastETLCutoffTime, GetLineageKey, etc.)
# using Delta tables in the target catalog.

def _entity_name(table_name):
    short = table_name.split('.')[-1] if '.' in table_name else table_name
    if short.endswith('_Staging'):
        short = short[:-len('_Staging')]
    return short


def get_last_etl_cutoff(table_name):
    entity = _entity_name(table_name)
    df = spark.sql(f"""
        SELECT `Cutoff Time` AS cutoff
        FROM `{_catalog}`.integration.etl_cutoff
        WHERE `Table Name` = '{entity}'
        ORDER BY `Cutoff Time` DESC
        LIMIT 1
    """)
    rows = df.collect()
    if rows:
        return rows[0]['cutoff'].strftime('%Y-%m-%d %H:%M:%S')
    return '1900-01-01 00:00:00'


def get_lineage_key(table_name, cutoff_time):
    entity = _entity_name(table_name)
    spark.sql(f"""
        INSERT INTO `{_catalog}`.integration.lineage (
            `Data Load Started`,
            `Table Name`,
            `Was Successful`,
            `Source System Cutoff Time`
        )
        VALUES (
            current_timestamp(),
            '{entity}',
            false,
            TIMESTAMP('{cutoff_time}')
        )
    """)
    row = spark.sql(f"""
        SELECT `Lineage Key` AS lk
        FROM `{_catalog}`.integration.lineage
        WHERE `Table Name` = '{entity}'
        ORDER BY `Lineage Key` DESC
        LIMIT 1
    """).collect()
    return row[0]['lk'] if row else 0


def complete_lineage(lineage_key):
    spark.sql(f"""
        UPDATE `{_catalog}`.integration.lineage
        SET `Data Load Completed` = current_timestamp(),
            `Was Successful` = true
        WHERE `Lineage Key` = {lineage_key}
    """)


def update_etl_cutoff(table_name, cutoff_time):
    entity = _entity_name(table_name)
    spark.sql(f"""
        MERGE INTO `{_catalog}`.integration.etl_cutoff AS t
        USING (SELECT '{entity}' AS `Table Name`, TIMESTAMP('{cutoff_time}') AS `Cutoff Time`) AS s
        ON t.`Table Name` = s.`Table Name`
        WHEN MATCHED THEN UPDATE SET t.`Cutoff Time` = s.`Cutoff Time`
        WHEN NOT MATCHED THEN INSERT (`Table Name`, `Cutoff Time`) VALUES (s.`Table Name`, s.`Cutoff Time`)
    """)

print('ETL helper functions defined: get_last_etl_cutoff, get_lineage_key, complete_lineage, update_etl_cutoff')


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load City Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetCityUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedCityData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [0]:
if _step_id in ("all", "load_city_dimension"):
    # [set_table_name] (CONTROL)
    table_name = "Integration.City_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql("""
    DELETE FROM Integration.City_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    # AI_REQUIRED
    #
    # Source procedure cannot be deterministically translated.
    #
    # Translation status:
    # UNSUPPORTED_PROCEDURAL_SQL
    #
    # Unsupported/complex features:
    # CURSOR, WHILE, TEMP_TABLE
    #
    # The original procedure is preserved in operation metadata.
    #
    # The migration agent must convert this procedure
    # into executable Databricks logic.


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.City"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.City"


    # [insert_target] (INSERT)
    # TODO: generate executable INSERT.
    #
    # Target:
    # Dimension.City
    #
    # Semantic analysis:
    # {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Dimension.City', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Customer Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_customer_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetCustomerUpdates` |
| Target Objects | `Integration.Customer_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Customer Dimension
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['TEMPORAL']
#
# Complexity signals:
# ['CURSOR', 'LOOP', 'TEMPORAL', 'TEMP_TABLE', 'UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetCustomerUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedCustomerData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [0]:
if _step_id in ("all", "load_customer_dimension"):
    # [set_table_name] (CONTROL)
    table_name = "Integration.Customer_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql("""
    DELETE FROM Integration.Customer_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    # AI_REQUIRED
    #
    # Source procedure cannot be deterministically translated.
    #
    # Translation status:
    # UNSUPPORTED_PROCEDURAL_SQL
    #
    # Unsupported/complex features:
    # CURSOR, WHILE, TEMP_TABLE
    #
    # The original procedure is preserved in operation metadata.
    #
    # The migration agent must convert this procedure
    # into executable Databricks logic.


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Customer"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Customer"


    # [insert_target] (INSERT)
    # TODO: generate executable INSERT.
    #
    # Target:
    # Dimension.Customer
    #
    # Semantic analysis:
    # {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Dimension.Customer', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Employee Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_employee_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetEmployeeUpdates` |
| Target Objects | `Integration.Employee_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- Source tables `Application.People` and `Application.People_Archive` are Delta tables accessible via Spark SQL and contain the columns PersonID, FullName, PreferredName, IsSalesperson, Photo, ValidFrom, ValidTo, and IsEmployee.
- `Integration.Employee_Staging` exists with a schema matching the columns produced by this transformation.
- The cut‑off parameters are supplied as strings that Spark can cast to timestamps.
- No additional business rules (e.g., filtering on other columns) are required beyond those expressed in the source procedure.


In [0]:
if _step_id in ("all", "load_employee_dimension"):
    from pyspark.sql import functions as F, Window

    def get_employee_updates(last_cutoff: str, new_cutoff: str):
        """Extract employee changes between two cut‑off timestamps and load them into the staging table.

        This implementation reproduces the logic of the source stored procedure
        `Integration.GetEmployeeUpdates` using Spark DataFrames on Databricks.

        Parameters
        ----------
        last_cutoff : str
            The previous ETL watermark (inclusive lower bound). Expected format
            parsable by Spark's ``timestamp`` type.
        new_cutoff : str
            The current ETL watermark (inclusive upper bound).
        """
        # Constant representing the far‑future "end of time" used in the source procedure
        END_OF_TIME = "9999-12-31 23:59:59.9999999"

        # ---------------------------------------------------------------------
        # 1. Load source tables (current and archive). They are expected to be
        #    Delta tables with the columns described in the migration context.
        # ---------------------------------------------------------------------
        people_cur = spark.table("Application.People")
        people_arc = spark.table("Application.People_Archive")

        # Union the two tables – this is the equivalent of the FOR SYSTEM_TIME AS OF
        # query that the source procedure uses.
        people_all = people_cur.unionByName(people_arc)

        # ---------------------------------------------------------------------
        # 2. Build the list of employee changes that occurred between the cut‑offs.
        #    The source uses a cursor over a UNION ALL of the archive and current
        #    tables, filtered on ValidFrom and IsEmployee.
        # ---------------------------------------------------------------------
        change_cur = (
            people_cur
            .filter(
                (F.col("ValidFrom") > F.lit(last_cutoff)) &
                (F.col("ValidFrom") <= F.lit(new_cutoff)) &
                (F.col("IsEmployee") != 0)
            )
            .select(
                F.col("PersonID").alias("WWI Employee ID"),
                F.col("ValidFrom")
            )
        )

        change_arc = (
            people_arc
            .filter(
                (F.col("ValidFrom") > F.lit(last_cutoff)) &
                (F.col("ValidFrom") <= F.lit(new_cutoff)) &
                (F.col("IsEmployee") != 0)
            )
            .select(
                F.col("PersonID").alias("WWI Employee ID"),
                F.col("ValidFrom")
            )
        )

        change_list = change_cur.unionByName(change_arc).orderBy("ValidFrom")

        # ---------------------------------------------------------------------
        # 3. For each change timestamp retrieve the employee row as it existed at
        #    that point in time. The temporal_context tells us to emulate
        #    "FOR SYSTEM_TIME AS OF @ValidFrom" by joining on the validity window:
        #        @ValidFrom >= ValidFrom AND @ValidFrom < ValidTo
        # ---------------------------------------------------------------------
        snapshot = (
            change_list.alias("c")
            .join(
                people_all.alias("p"),
                (F.col("p.PersonID") == F.col("c.WWI Employee ID")) &
                (F.col("p.ValidFrom") <= F.col("c.ValidFrom")) &
                (F.col("c.ValidFrom") < F.col("p.ValidTo")),
                "inner"
            )
            .select(
                F.col("c.WWI Employee ID"),
                F.col("p.FullName").alias("Employee"),
                F.col("p.PreferredName").alias("Preferred Name"),
                F.col("p.IsSalesperson").alias("Is Salesperson"),
                F.col("p.Photo"),
                F.col("c.ValidFrom").alias("Valid From"),
                F.lit(END_OF_TIME).alias("Valid To")
            )
        )

        # ---------------------------------------------------------------------
        # 4. Compute the correct "Valid To" for each employee version. The source
        #    procedure sets Valid To to the next Valid From for the same employee,
        #    or to the end‑of‑time constant if no later version exists.
        # ---------------------------------------------------------------------
        w = Window.partitionBy("WWI Employee ID").orderBy("Valid From")
        result = (
            snapshot
            .withColumn("next_valid_from", F.lead("Valid From").over(w))
            .withColumn(
                "Valid To",
                F.when(F.col("next_valid_from").isNotNull(), F.col("next_valid_from")).otherwise(F.lit(END_OF_TIME))
            )
            .drop("next_valid_from")
        )

        # ---------------------------------------------------------------------
        # 5. Load the result into the staging table used by the downstream dimension
        #    load. The destination procedure expects the data to be present in
        #    `Integration.Employee_Staging`.
        # ---------------------------------------------------------------------
        result.write.mode("append").saveAsTable("Integration.Employee_Staging")

        return result

    # Example usage (uncomment when running in a notebook):
    # df = get_employee_updates('2023-01-01 00:00:00', '2023-01-31 23:59:59')
    # df.show()

## Load Movement Fact

| Field | Value |
| --- | --- |
| Step ID | `load_movement_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetMovementUpdates` |
| Target Objects | `Integration.Movement_Staging` |


In [0]:
if _step_id in ("all", "load_movement_fact"):
    # [set_table_name] (CONTROL)
    table_name = "Integration.Movement_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql("""
    DELETE FROM Integration.Movement_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    source_df = spark.sql("""
    SELECT CAST(sit.TransactionOccurredWhen AS date) AS `Date Key`,
               sit.StockItemTransactionID AS `WWI Stock Item Transaction ID`,
               sit.InvoiceID AS `WWI Invoice ID`,
               sit.PurchaseOrderID AS `WWI Purchase Order ID`,
               CAST(sit.Quantity AS int) AS Quantity,
               sit.StockItemID AS `WWI Stock Item ID`,
               sit.CustomerID AS `WWI Customer ID`,
               sit.SupplierID AS `WWI Supplier ID`,
               sit.TransactionTypeID AS `WWI Transaction Type ID`,
               sit.TransactionOccurredWhen AS `Transaction Occurred When`
        FROM Warehouse.StockItemTransactions AS sit
        WHERE sit.LastEditedWhen > :LastCutoff
        AND sit.LastEditedWhen <= :NewCutoff
        ORDER BY sit.StockItemTransactionID
    """)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Stock Item", "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Stock Item", "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type"


    # [merge_target] (MERGE)
    # TODO: generate executable Delta MERGE.
    #
    # Target:
    # Fact.Movement
    #
    # Semantic analysis:
    # {'operation': 'MERGE_TARGET', 'type': 'MERGE', 'description': 'Merge staged records into the target fact or dimension table.', 'target': 'Fact.Movement', 'features': ['WHEN_MATCHED', 'WHEN_NOT_MATCHED', 'UPDATE', 'INSERT']}
    #
    # The source procedure definition is preserved
    # in operation metadata for deterministic generation
    # or AI-assisted refinement.


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Order Fact

| Field | Value |
| --- | --- |
| Step ID | `load_order_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetOrderUpdates` |
| Target Objects | `Integration.Order_Staging` |


In [0]:
if _step_id in ("all", "load_order_fact"):
    # [set_table_name] (CONTROL)
    table_name = "Integration.Order_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql("""
    DELETE FROM Integration.Order_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    source_df = spark.sql("""
    SELECT CAST(o.OrderDate AS date) AS `Order Date Key`,
               CAST(ol.PickingCompletedWhen AS date) AS `Picked Date Key`,
               o.OrderID AS `WWI Order ID`,
               o.BackorderOrderID AS `WWI Backorder ID`,
               ol.`Description`,
               pt.PackageTypeName AS Package,
               ol.Quantity AS Quantity,
               ol.UnitPrice AS `Unit Price`,
               ol.TaxRate AS `Tax Rate`,
               ROUND(ol.Quantity * ol.UnitPrice, 2) AS `Total Excluding Tax`,
               ROUND(ol.Quantity * ol.UnitPrice * ol.TaxRate / 100.0, 2) AS `Tax Amount`,
               ROUND(ol.Quantity * ol.UnitPrice, 2) + ROUND(ol.Quantity * ol.UnitPrice * ol.TaxRate / 100.0, 2) AS `Total Including Tax`,
               c.DeliveryCityID AS `WWI City ID`,
               c.CustomerID AS `WWI Customer ID`,
               ol.StockItemID AS `WWI Stock Item ID`,
               o.SalespersonPersonID AS `WWI Salesperson ID`,
               o.PickedByPersonID AS `WWI Picker ID`,
               CASE WHEN ol.LastEditedWhen > o.LastEditedWhen THEN ol.LastEditedWhen ELSE o.LastEditedWhen END AS `Last Modified When`
        FROM Sales.Orders AS o
        INNER JOIN Sales.OrderLines AS ol
        ON o.OrderID = ol.OrderID
        INNER JOIN Warehouse.PackageTypes AS pt
        ON ol.PackageTypeID = pt.PackageTypeID
        INNER JOIN Sales.Customers AS c
        ON c.CustomerID = o.CustomerID
        WHERE CASE WHEN ol.LastEditedWhen > o.LastEditedWhen THEN ol.LastEditedWhen ELSE o.LastEditedWhen END > :LastCutoff
        AND CASE WHEN ol.LastEditedWhen > o.LastEditedWhen THEN ol.LastEditedWhen ELSE o.LastEditedWhen END <= :NewCutoff
        ORDER BY o.OrderID
    """)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"


    # [delete_existing_target] (DELETE)
    # TODO: generate executable DELETE.
    #
    # Target:
    # Fact.Order
    #
    # Semantic analysis:
    # {'operation': 'DELETE_EXISTING_TARGET', 'type': 'DELETE', 'description': 'Remove existing target records affected by the incoming staging records.', 'target': 'Fact.Order', 'features': ['DELETE', 'STAGING_KEY_FILTER']}


    # [insert_target] (INSERT)
    # TODO: generate executable INSERT.
    #
    # Target:
    # Fact.Order
    #
    # Semantic analysis:
    # {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Order', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Payment Method Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_payment_method_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetPaymentMethodUpdates` |
| Target Objects | `Integration.PaymentMethod_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- Source tables Application.PaymentMethods and Application.PaymentMethods_Archive are Delta tables that contain the columns PaymentMethodID, PaymentMethodName, ValidFrom, ValidTo.
- The EndOfTime constant '9999-12-31 23:59:59.9999999' is compatible with the target schema.
- Integration.PaymentMethod_Staging, Dimension.Payment Method, Integration.Lineage, and Integration.ETL Cutoff already exist as Delta tables with the columns referenced in the original SQL procedures.
- Databricks runtime supports Spark SQL UPDATE statements on Delta tables (i.e., Delta Lake version >= 0.8).
- The workflow that calls this function will handle clearing the staging table after a successful run if required.


In [0]:
if _step_id in ("all", "load_payment_method_dimension"):
    import pyspark.sql.functions as F
    from pyspark.sql import Window, SparkSession

    # ---------------------------------------------------------------------------
    # Load Payment Method Dimension
    # ---------------------------------------------------------------------------
    # This function implements the full logical step "load_payment_method_dimension"
    #   * Extracts incremental payment‑method changes between two cut‑off timestamps
    #   * Resolves the point‑in‑time view using row‑level ValidFrom/ValidTo columns
    #   * Computes the correct ValidTo for each change (next ValidFrom or EndOfTime)
    #   * Writes the result to the staging table Integration.PaymentMethod_Staging
    #   * Merges the staged rows into the target dimension table, closes off prior
    #     rows, updates lineage and ETL cutoff information.
    # ---------------------------------------------------------------------------

    def load_payment_method_dimension(last_cutoff: str, new_cutoff: str, spark: SparkSession = None):
        """Execute the Load Payment Method Dimension step.

        Parameters
        ----------
        last_cutoff: str
            ISO‑8601 timestamp representing the previous successful ETL watermark.
        new_cutoff: str
            ISO‑8601 timestamp representing the current ETL watermark.
        spark: SparkSession, optional
            Existing Spark session; if omitted, ``SparkSession.builder.getOrCreate()`` is used.
        """
        if spark is None:
            spark = SparkSession.builder.getOrCreate()

        # ---------------------------------------------------------------------
        # 1. Constants & helper tables
        # ---------------------------------------------------------------------
        END_OF_TIME = "9999-12-31 23:59:59.9999999"

        # Source tables (assumed to be Delta tables with system‑versioned columns)
        CURRENT_TABLE = "Application.PaymentMethods"
        ARCHIVE_TABLE = "Application.PaymentMethods_Archive"

        # ---------------------------------------------------------------------
        # 2. Build the change list (equivalent to the cursor in the original proc)
        # ---------------------------------------------------------------------
        change_list_sql = f"""
            SELECT PaymentMethodID, ValidFrom
            FROM {ARCHIVE_TABLE}
            WHERE ValidFrom > TIMESTAMP('{last_cutoff}')
              AND ValidFrom <= TIMESTAMP('{new_cutoff}')
            UNION ALL
            SELECT PaymentMethodID, ValidFrom
            FROM {CURRENT_TABLE}
            WHERE ValidFrom > TIMESTAMP('{last_cutoff}')
              AND ValidFrom <= TIMESTAMP('{new_cutoff}')
            ORDER BY ValidFrom
        """
        change_df = spark.sql(change_list_sql)

        # ---------------------------------------------------------------------
        # 3. Resolve the point‑in‑time view for each change using row‑level validity
        #    (FOR SYSTEM_TIME AS OF @ValidFrom  =>  ValidFrom <= @ValidFrom < ValidTo)
        # ---------------------------------------------------------------------
        # Union of current and archive tables provides the full history.
        history_sql = f"""
            SELECT PaymentMethodID,
                   PaymentMethodName,
                   ValidFrom,
                   ValidTo
            FROM {CURRENT_TABLE}
            UNION ALL
            SELECT PaymentMethodID,
                   PaymentMethodName,
                   ValidFrom,
                   ValidTo
            FROM {ARCHIVE_TABLE}
        """
        history_df = spark.sql(history_sql)

        # Join change list with history on ID and timestamp window
        as_of_df = (
            change_df.alias("cl")
            .join(
                history_df.alias("h"),
                (F.col("cl.PaymentMethodID") == F.col("h.PaymentMethodID"))
                & (F.col("cl.ValidFrom") >= F.col("h.ValidFrom"))
                & (F.col("cl.ValidFrom") < F.col("h.ValidTo")),
                "inner",
            )
            .select(
                F.col("h.PaymentMethodID").alias("WWI Payment Method ID"),
                F.col("h.PaymentMethodName").alias("Payment Method"),
                F.col("h.ValidFrom").alias("Valid From"),
                F.col("h.ValidTo").alias("Valid To"),
            )
        )

        # ---------------------------------------------------------------------
        # 4. Re‑compute Valid To based on the next Valid From for the same ID
        # ---------------------------------------------------------------------
        w = Window.partitionBy("WWI Payment Method ID").orderBy("Valid From")
        staged_df = (
            as_of_df
            .withColumn(
                "Valid To",
                F.when(
                    F.lead("Valid From").over(w).isNotNull(),
                    F.lead("Valid From").over(w)
                ).otherwise(F.lit(END_OF_TIME))
            )
        )

        # ---------------------------------------------------------------------
        # 5. Write to staging table (append – the surrounding workflow will clear it
        #    before the next run)
        # ---------------------------------------------------------------------
        staged_df.write.format("delta").mode("append").saveAsTable("Integration.PaymentMethod_Staging")

        # ---------------------------------------------------------------------
        # 6. Migrate staged data into the dimension table (logic from
        #    Integration.MigrateStagedPaymentMethodData)
        # ---------------------------------------------------------------------
        # 6.1 Retrieve the current lineage key for this load
        lineage_key_sql = """
            SELECT LineageKey
            FROM Integration.Lineage
            WHERE `Table Name` = 'payment_method' AND `Data Load Completed` IS NULL
            ORDER BY LineageKey DESC
            LIMIT 1
        """
        lineage_key_row = spark.sql(lineage_key_sql).collect()
        if not lineage_key_row:
            raise RuntimeError("No pending lineage record found for 'payment_method'.")
        lineage_key = lineage_key_row[0]["LineageKey"]

        # 6.2 Close off existing rows in the dimension that are still open (Valid To = EndOfTime)
        rows_to_close_sql = """
            SELECT `WWI Payment Method ID`, MIN(`Valid From`) AS `Valid From`
            FROM Integration.PaymentMethod_Staging
            GROUP BY `WWI Payment Method ID`
        """
        rows_to_close_df = spark.sql(rows_to_close_sql)
        rows_to_close_df.createOrReplaceTempView("RowsToCloseOff")

        close_off_sql = f"""
            UPDATE Dimension.payment_method AS pm
            SET pm.`Valid To` = r.`Valid From`
            FROM RowsToCloseOff r
            WHERE pm.`WWI Payment Method ID` = r.`WWI Payment Method ID`
              AND pm.`Valid To` = TIMESTAMP('{END_OF_TIME}')
        """
        spark.sql(close_off_sql)

        # 6.3 Insert new rows from staging
        insert_sql = f"""
            INSERT INTO Dimension.payment_method (
                `WWI Payment Method ID`, `Payment Method`, `Valid From`, `Valid To`, `Lineage Key`
            )
            SELECT `WWI Payment Method ID`, `Payment Method`, `Valid From`, `Valid To`, {lineage_key}
            FROM Integration.PaymentMethod_Staging
        """
        spark.sql(insert_sql)

        # 6.4 Mark lineage as completed
        update_lineage_sql = f"""
            UPDATE Integration.Lineage
            SET `Data Load Completed` = CURRENT_TIMESTAMP(),
                `Was Successful` = 1
            WHERE LineageKey = {lineage_key}
        """
        spark.sql(update_lineage_sql)

        # 6.5 Advance ETL cutoff for this table
        update_cutoff_sql = f"""
            UPDATE Integration.`ETL Cutoff`
            SET `Cutoff Time` = (
                SELECT `Source System Cutoff Time`
                FROM Integration.Lineage
                WHERE LineageKey = {lineage_key}
            )
            WHERE `Table Name` = 'payment_method'
        """
        spark.sql(update_cutoff_sql)

        # ---------------------------------------------------------------------
        # 7. End of step – callers can now clear the staging table if required.
        # ---------------------------------------------------------------------
        return {
            "staged_rows": staged_df.count(),
            "lineage_key": lineage_key
        }

    # Example usage (uncomment for real runs):
    # result = load_payment_method_dimension('2023-01-01T00:00:00', '2023-01-31T23:59:59')
    # print(result)


## Load Purchase Fact

| Field | Value |
| --- | --- |
| Step ID | `load_purchase_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetPurchaseUpdates` |
| Target Objects | `Integration.Purchase_Staging` |


In [0]:
if _step_id in ("all", "load_purchase_fact"):
    # [set_table_name] (CONTROL)
    table_name = "Integration.Purchase_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql("""
    DELETE FROM Integration.Purchase_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    source_df = spark.sql("""
    SELECT CAST(po.OrderDate AS date) AS `Date Key`,
               po.PurchaseOrderID AS `WWI Purchase Order ID`,
               pol.OrderedOuters AS `Ordered Outers`,
               pol.OrderedOuters * si.QuantityPerOuter AS `Ordered Quantity`,
               pol.ReceivedOuters AS `Received Outers`,
               pt.PackageTypeName AS Package,
               pol.IsOrderLineFinalized AS `Is Order Finalized`,
               po.SupplierID AS `WWI Supplier ID`,
               pol.StockItemID AS `WWI Stock Item ID`,
               CASE WHEN pol.LastEditedWhen > po.LastEditedWhen THEN pol.LastEditedWhen ELSE po.LastEditedWhen END AS `Last Modified When`
        FROM Purchasing.PurchaseOrders AS po
        INNER JOIN Purchasing.PurchaseOrderLines AS pol
        ON po.PurchaseOrderID = pol.PurchaseOrderID
        INNER JOIN Warehouse.StockItems AS si
        ON pol.StockItemID = si.StockItemID
        INNER JOIN Warehouse.PackageTypes AS pt
        ON pol.PackageTypeID = pt.PackageTypeID
        WHERE CASE WHEN pol.LastEditedWhen > po.LastEditedWhen THEN pol.LastEditedWhen ELSE po.LastEditedWhen END > :LastCutoff
        AND CASE WHEN pol.LastEditedWhen > po.LastEditedWhen THEN pol.LastEditedWhen ELSE po.LastEditedWhen END <= :NewCutoff
        ORDER BY po.PurchaseOrderID
    """)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Supplier", "Dimension.Stock Item"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Supplier", "Dimension.Stock Item"


    # [delete_existing_target] (DELETE)
    # TODO: generate executable DELETE.
    #
    # Target:
    # Fact.Purchase
    #
    # Semantic analysis:
    # {'operation': 'DELETE_EXISTING_TARGET', 'type': 'DELETE', 'description': 'Remove existing target records affected by the incoming staging records.', 'target': 'Fact.Purchase', 'features': ['DELETE', 'STAGING_KEY_FILTER']}


    # [insert_target] (INSERT)
    # TODO: generate executable INSERT.
    #
    # Target:
    # Fact.Purchase
    #
    # Semantic analysis:
    # {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Purchase', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Sale Fact

| Field | Value |
| --- | --- |
| Step ID | `load_sale_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetSaleUpdates` |
| Target Objects | `Integration.Sale_Staging` |


In [0]:
if _step_id in ("all", "load_sale_fact"):
    # [set_table_name] (CONTROL)
    table_name = "Integration.Sale_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql("""
    DELETE FROM Integration.Sale_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    source_df = spark.sql("""
    SELECT CAST(i.InvoiceDate AS date) AS `Invoice Date Key`,
               CAST(i.ConfirmedDeliveryTime AS date) AS `Delivery Date Key`,
               i.InvoiceID AS `WWI Invoice ID`,
               il.`Description`,
               pt.PackageTypeName AS Package,
               il.Quantity,
               il.UnitPrice AS `Unit Price`,
               il.TaxRate AS `Tax Rate`,
               il.ExtendedPrice - il.TaxAmount AS `Total Excluding Tax`,
               il.TaxAmount AS `Tax Amount`,
               il.LineProfit AS Profit,
               il.ExtendedPrice AS `Total Including Tax`,
               CASE WHEN si.IsChillerStock = 0 THEN il.Quantity ELSE 0 END AS `Total Dry Items`,
               CASE WHEN si.IsChillerStock <> 0 THEN il.Quantity ELSE 0 END AS `Total Chiller Items`,
               c.DeliveryCityID AS `WWI City ID`,
               i.CustomerID AS `WWI Customer ID`,
               i.BillToCustomerID AS `WWI Bill To Customer ID`,
               il.StockItemID AS `WWI Stock Item ID`,
               i.SalespersonPersonID AS `WWI Saleperson ID`,
               CASE WHEN il.LastEditedWhen > i.LastEditedWhen THEN il.LastEditedWhen ELSE i.LastEditedWhen END AS `Last Modified When`
        FROM Sales.Invoices AS i
        INNER JOIN Sales.InvoiceLines AS il
        ON i.InvoiceID = il.InvoiceID
        INNER JOIN Warehouse.StockItems AS si
        ON il.StockItemID = si.StockItemID
        INNER JOIN Warehouse.PackageTypes AS pt
        ON il.PackageTypeID = pt.PackageTypeID
        INNER JOIN Sales.Customers AS c
        ON i.CustomerID = c.CustomerID
        INNER JOIN Sales.Customers AS bt
        ON i.BillToCustomerID = bt.CustomerID
        WHERE CASE WHEN il.LastEditedWhen > i.LastEditedWhen THEN il.LastEditedWhen ELSE i.LastEditedWhen END > :LastCutoff
        AND CASE WHEN il.LastEditedWhen > i.LastEditedWhen THEN il.LastEditedWhen ELSE i.LastEditedWhen END <= :NewCutoff
        ORDER BY i.InvoiceID, il.InvoiceLineID
    """)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.City", "Dimension.Customer", "Dimension.Stock Item", "Dimension.Employee"


    # [delete_existing_target] (DELETE)
    # TODO: generate executable DELETE.
    #
    # Target:
    # Fact.Sale
    #
    # Semantic analysis:
    # {'operation': 'DELETE_EXISTING_TARGET', 'type': 'DELETE', 'description': 'Remove existing target records affected by the incoming staging records.', 'target': 'Fact.Sale', 'features': ['DELETE', 'STAGING_KEY_FILTER']}


    # [insert_target] (INSERT)
    # TODO: generate executable INSERT.
    #
    # Target:
    # Fact.Sale
    #
    # Semantic analysis:
    # {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Sale', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Stock Holding Fact

| Field | Value |
| --- | --- |
| Step ID | `load_stock_holding_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `DETERMINISTIC` |
| Dependencies | None |
| Source Objects | `Integration.GetStockHoldingUpdates` |
| Target Objects | `Integration.StockHolding_Staging` |


In [0]:
if _step_id in ("all", "load_stock_holding_fact"):
    # [set_table_name] (CONTROL)
    table_name = "Integration.StockHolding_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql("""
    DELETE FROM Integration.StockHolding_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    source_df = spark.sql("""
    SELECT sih.QuantityOnHand AS `Quantity On Hand`,
               sih.BinLocation AS `Bin Location`,
               sih.LastStocktakeQuantity AS `Last Stocktake Quantity`,
               sih.LastCostPrice AS `Last Cost Price`,
               sih.ReorderLevel AS `Reorder Level`,
               sih.TargetStockLevel AS `Target Stock Level`,
               sih.StockItemID AS `WWI Stock Item ID`
        FROM Warehouse.StockItemHoldings AS sih
        ORDER BY sih.StockItemID
    """)


    # [truncate_target] (TRUNCATE)
    spark.sql("""
    TRUNCATE TABLE Fact.Stock Holding
    """)


    # [insert_target] (INSERT)
    # TODO: generate executable INSERT.
    #
    # Target:
    # Fact.Stock Holding
    #
    # Semantic analysis:
    # {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Stock Holding', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Stock Item Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_stock_item_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetStockItemUpdates` |
| Target Objects | `Integration.StockItem_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- Archive tables for PackageTypes and Colors exist with the naming pattern `<schema>.<table>_Archive` and contain the same schema (including ValidFrom/ValidTo) as the current tables.
- Integration.StockItem_Staging is a pre‑created Delta table with a schema matching the columns produced by the source procedure.
- The column `Tax Rate` in the original procedure is populated from `LeadTimeDays` – this appears to be a copy‑paste error in the source; the migration preserves the exact behaviour.
- The driver memory can hold the list of change points; the number of distinct StockItem change timestamps is expected to be modest enough for a Python‑side loop.


In [0]:
if _step_id in ("all", "load_stock_item_dimension"):
    import pyspark.sql.functions as F
    import pyspark.sql.window as W
    from datetime import datetime

    END_OF_TIME = "9999-12-31 23:59:59.9999999"

    def _snapshot_at_timestamp(df_current, df_archive, timestamp_col, valid_from_col="ValidFrom", valid_to_col="ValidTo"):
        """Return a DataFrame representing the union of the current and archive tables filtered to rows valid at the supplied timestamp.
    
        Parameters
        ----------
        df_current: DataFrame
            The live (current) table.
        df_archive: DataFrame
            The archive table (historical rows).
        timestamp_col: Column (or literal) containing the point‑in‑time timestamp.
        valid_from_col: Name of the column that stores the start of validity.
        valid_to_col: Name of the column that stores the end of validity.
        """
        # Union current and archive rows
        df_union = df_current.unionByName(df_archive)
        # Keep rows where timestamp is within the validity window
        return df_union.filter(
            (F.col(valid_from_col) <= timestamp_col) & (F.col(valid_to_col) > timestamp_col)
        )

    def load_stock_item_dimension(spark, last_cutoff: datetime, new_cutoff: datetime):
        """Extract changed StockItem rows between two cut‑off timestamps and write them to the staging table.
    
        This function reproduces the behaviour of the source stored procedure
        `Integration.GetStockItemUpdates` using Spark DataFrames on Databricks.
        """
        # ---------------------------------------------------------------------
        # 1. Identify StockItem change points (StockItemID, ValidFrom) within the window
        # ---------------------------------------------------------------------
        stock_items_cur = spark.table("Warehouse.StockItems")
        stock_items_arc = spark.table("Warehouse.StockItems_Archive")

        changes_cur = stock_items_cur.filter(
            (F.col("ValidFrom") > F.lit(last_cutoff)) & (F.col("ValidFrom") <= F.lit(new_cutoff))
        ).select(F.col("StockItemID").alias("WWI_Stock_Item_ID"), F.col("ValidFrom"))

        changes_arc = stock_items_arc.filter(
            (F.col("ValidFrom") > F.lit(last_cutoff)) & (F.col("ValidFrom") <= F.lit(new_cutoff))
        ).select(F.col("StockItemID").alias("WWI_Stock_Item_ID"), F.col("ValidFrom"))

        # UNION ALL and order by ValidFrom
        change_points = changes_cur.unionByName(changes_arc).orderBy("ValidFrom")

        # Collect change points to driver for procedural iteration (preserves original cursor/loop semantics)
        change_list = change_points.select("WWI_Stock_Item_ID", "ValidFrom").collect()

        # ---------------------------------------------------------------------
        # 2. For each change point, materialise the snapshot of related tables at that timestamp
        # ---------------------------------------------------------------------
        rows = []
        for row in change_list:
            stock_item_id = row["WWI_Stock_Item_ID"]
            valid_from_ts = row["ValidFrom"]
            ts_lit = F.lit(valid_from_ts)

            # Snapshot of StockItems at @ValidFrom
            stock_item_snapshot = _snapshot_at_timestamp(
                spark.table("Warehouse.StockItems"),
                spark.table("Warehouse.StockItems_Archive"),
                ts_lit
            ).filter(F.col("StockItemID") == stock_item_id)

            # Snapshot of PackageTypes (unit) at @ValidFrom
            unit_pkg_snapshot = _snapshot_at_timestamp(
                spark.table("Warehouse.PackageTypes"),
                spark.table("Warehouse.PackageTypes_Archive"),
                ts_lit
            )

            # Snapshot of PackageTypes (outer) at @ValidFrom
            outer_pkg_snapshot = _snapshot_at_timestamp(
                spark.table("Warehouse.PackageTypes"),
                spark.table("Warehouse.PackageTypes_Archive"),
                ts_lit
            )

            # Snapshot of Colors at @ValidFrom (LEFT OUTER – may be null)
            colors_snapshot = _snapshot_at_timestamp(
                spark.table("Warehouse.Colors"),
                spark.table("Warehouse.Colors_Archive"),
                ts_lit
            )

            # Join the snapshots to produce the required columns
            joined = (
                stock_item_snapshot.alias("si")
                .join(
                    unit_pkg_snapshot.alias("spt"),
                    F.col("si.UnitPackageID") == F.col("spt.PackageTypeID"),
                    "inner",
                )
                .join(
                    outer_pkg_snapshot.alias("bpt"),
                    F.col("si.OuterPackageID") == F.col("bpt.PackageTypeID"),
                    "inner",
                )
                .join(
                    colors_snapshot.alias("c"),
                    F.col("si.ColorID") == F.col("c.ColorID"),
                    "left",
                )
                .select(
                    F.col("si.StockItemID").alias("WWI Stock Item ID"),
                    F.col("si.StockItemName").alias("Stock Item"),
                    F.col("c.ColorName").alias("Color"),
                    F.col("spt.PackageTypeName").alias("Selling Package"),
                    F.col("bpt.PackageTypeName").alias("Buying Package"),
                    F.col("si.Brand").alias("Brand"),
                    F.col("si.Size").alias("Size"),
                    F.col("si.LeadTimeDays").alias("Lead Time Days"),
                    F.col("si.QuantityPerOuter").alias("Quantity Per Outer"),
                    F.col("si.IsChillerStock").alias("Is Chiller Stock"),
                    F.col("si.Barcode").alias("Barcode"),
                    F.col("si.LeadTimeDays").alias("Tax Rate"),  # NOTE: original proc uses LeadTimeDays for Tax Rate – preserving semantics
                    F.col("si.UnitPrice").alias("Unit Price"),
                    F.col("si.RecommendedRetailPrice").alias("Recommended Retail Price"),
                    F.col("si.TypicalWeightPerUnit").alias("Typical Weight Per Unit"),
                    F.col("si.Photo").alias("Photo"),
                    F.col("si.ValidFrom").alias("Valid From"),
                    F.col("si.ValidTo").alias("Valid To"),
                )
            )

            # Collect the single row (the source proc expects exactly one row per change point)
            for result_row in joined.collect():
                rows.append(result_row.asDict())

        # ---------------------------------------------------------------------
        # 3. Build a DataFrame from the collected rows (equivalent to #StockItemChanges temp table)
        # ---------------------------------------------------------------------
        if not rows:
            # No changes – create an empty DataFrame with the correct schema
            schema = "[WWI Stock Item ID] INT, [Stock Item] STRING, Color STRING, [Selling Package] STRING, " \
                     "[Buying Package] STRING, Brand STRING, Size STRING, [Lead Time Days] INT, " \
                     "[Quantity Per Outer] INT, [Is Chiller Stock] BOOLEAN, Barcode STRING, " \
                     "[Tax Rate] DECIMAL(18,3), [Unit Price] DECIMAL(18,2), " \
                     "[Recommended Retail Price] DECIMAL(18,2), [Typical Weight Per Unit] DECIMAL(18,3), " \
                     "Photo BINARY, [Valid From] TIMESTAMP, [Valid To] TIMESTAMP"
            stock_changes_df = spark.createDataFrame([], schema)
        else:
            stock_changes_df = spark.createDataFrame(rows)

        # ---------------------------------------------------------------------
        # 4. Compute the correct [Valid To] for each row (next Valid From per WWI Stock Item ID)
        # ---------------------------------------------------------------------
        w = W.Window.partitionBy("WWI Stock Item ID").orderBy("Valid From")
        stock_changes_df = stock_changes_df.withColumn(
            "NextValidFrom", F.lead("Valid From").over(w)
        ).withColumn(
            "Valid To",
            F.coalesce(F.col("NextValidFrom"), F.lit(END_OF_TIME).cast("timestamp"))
        ).drop("NextValidFrom")

        # ---------------------------------------------------------------------
        # 5. Write the result to the staging table Integration.StockItem_Staging
        # ---------------------------------------------------------------------
        # Assuming the staging table already exists as a Delta table.
        stock_changes_df.write.mode("append").format("delta").saveAsTable("Integration.StockItem_Staging")

        # The original procedure returns 0 on success – we mimic that with a Python return value.
        return 0

    # Example usage (to be removed or adapted in production):
    # from datetime import datetime
    # spark = SparkSession.builder.getOrCreate()
    # load_stock_item_dimension(spark, datetime(2023, 1, 1), datetime(2023, 1, 31))


## Load Supplier Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_supplier_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetSupplierUpdates` |
| Target Objects | `Integration.Supplier_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- Source tables Purchasing.Suppliers, Purchasing.Suppliers_Archive, Purchasing.SupplierCategories, Purchasing.SupplierCategories_Archive, and Application.People exist as Delta tables in Databricks.
- All temporal tables contain ValidFrom and ValidTo columns that define the validity window for each row.
- Application.People is not versioned; the current version is sufficient for the join.
- Integration.ETL_Cutoff table contains a row with Table Name = 'Supplier' and a Cutoff Time column representing the last successful load.
- The staging table Integration.Supplier_Staging already exists with a schema matching the SELECT list of the source procedure.
- The orchestration layer will invoke the subsequent destination procedure Integration.MigrateStagedSupplierData after this step.


In [0]:
if _step_id in ("all", "load_supplier_dimension"):
    # Databricks (PySpark) implementation of Integration.GetSupplierUpdates
    # This code reproduces the logic of the source stored procedure and writes the result
    # to the staging table Integration.Supplier_Staging.
    # It assumes that the source tables are available as Delta tables with the same
    # schema as in the source system and that temporal history is stored using the
    # ValidFrom / ValidTo columns together with archive tables following the pattern
    #   <schema>.<table>_Archive.
    # The People table is assumed to be non‑temporal (only the current version is used).

    from pyspark.sql import SparkSession, functions as F, Window

    spark = SparkSession.builder.getOrCreate()

    # ---------------------------------------------------------------------------
    # Parameters / constants (these would normally be supplied by the orchestration layer)
    # ---------------------------------------------------------------------------
    # Retrieve the last successful cutoff for the Supplier dimension
    # (Integration.ETL Cutoff table is expected to have columns: Table Name, Cutoff Time)
    last_cutoff_row = spark.table("Integration.ETL_Cutoff") \
        .filter(F.col("Table Name") == "Supplier") \
        .select(F.col("Cutoff Time").alias("last_cutoff")) \
        .orderBy(F.col("last_cutoff").desc()) \
        .limit(1) \
        .collect()

    if not last_cutoff_row:
        raise RuntimeError("Unable to determine last ETL cutoff for Supplier dimension.")

    last_cutoff = last_cutoff_row[0]["last_cutoff"]
    # New cutoff is the current timestamp at execution time
    new_cutoff = F.current_timestamp()

    END_OF_TIME = "9999-12-31 23:59:59.9999999"
    INITIAL_LOAD_DATE = "2013-01-01"

    # ---------------------------------------------------------------------------
    # Helper to obtain a point‑in‑time snapshot of a temporal table
    # ---------------------------------------------------------------------------
    def snapshot(df, ts_col):
        """Return rows that are valid at the timestamp given by ts_col (Column)."""
        return df.filter((F.col("ValidFrom") <= ts_col) & (F.col("ValidTo") > ts_col))

    # ---------------------------------------------------------------------------
    # Load source tables (current and archive) – they must contain ValidFrom/ValidTo
    # ---------------------------------------------------------------------------
    suppliers_cur = spark.table("Purchasing.Suppliers")
    suppliers_arc = spark.table("Purchasing.Suppliers_Archive")
    categories_cur = spark.table("Purchasing.SupplierCategories")
    categories_arc = spark.table("Purchasing.SupplierCategories_Archive")
    people = spark.table("Application.People")  # assumed non‑temporal

    # Union current and archive to obtain the full temporal history
    suppliers_hist = suppliers_cur.unionByName(suppliers_arc)
    categories_hist = categories_cur.unionByName(categories_arc)

    # ---------------------------------------------------------------------------
    # 1) Supplier Category changes (the first cursor in the original proc)
    # ---------------------------------------------------------------------------
    cat_changes = categories_hist.filter(
        (F.col("ValidFrom") > F.lit(last_cutoff)) &
        (F.col("ValidFrom") <= F.lit(new_cutoff)) &
        (F.col("ValidFrom") != F.lit(INITIAL_LOAD_DATE))
    ).select(
        F.col("SupplierCategoryID").alias("cat_id"),
        F.col("ValidFrom").alias("change_ts")
    ).orderBy("change_ts")

    # Join suppliers and categories as they were at each change timestamp
    cat_change_rows = cat_changes.alias("cc") \
        .join(
            snapshot(suppliers_hist, F.col("cc.change_ts")).alias("s"),
            F.col("s.SupplierCategoryID") == F.col("cc.cat_id"),
            "inner"
        ) \
        .join(
            snapshot(categories_hist, F.col("cc.change_ts")).alias("sc"),
            (F.col("sc.SupplierCategoryID") == F.col("cc.cat_id")) &
            (F.col("sc.ValidFrom") <= F.col("cc.change_ts")) &
            (F.col("sc.ValidTo") > F.col("cc.change_ts")),
            "inner"
        ) \
        .join(
            people.alias("p"),
            F.col("s.PrimaryContactPersonID") == F.col("p.PersonID"),
            "inner"
        ) \
        .select(
            F.col("s.SupplierID").alias("WWI Supplier ID"),
            F.col("s.SupplierName").alias("Supplier"),
            F.col("sc.SupplierCategoryName").alias("Category"),
            F.col("p.FullName").alias("Primary Contact"),
            F.col("s.SupplierReference").alias("Supplier Reference"),
            F.col("s.PaymentDays").alias("Payment Days"),
            F.col("s.DeliveryPostalCode").alias("Postal Code"),
            F.col("cc.change_ts").alias("Valid From"),
            F.col("s.ValidTo").alias("Valid To")
        )

    # ---------------------------------------------------------------------------
    # 2) Supplier changes (the second cursor in the original proc)
    # ---------------------------------------------------------------------------
    supplier_changes = suppliers_hist.filter(
        (F.col("ValidFrom") > F.lit(last_cutoff)) &
        (F.col("ValidFrom") <= F.lit(new_cutoff))
    ).select(
        F.col("SupplierID").alias("sup_id"),
        F.col("ValidFrom").alias("change_ts")
    ).orderBy("change_ts")

    supplier_change_rows = supplier_changes.alias("scu") \
        .join(
            snapshot(suppliers_hist, F.col("scu.change_ts")).alias("s"),
            F.col("s.SupplierID") == F.col("scu.sup_id"),
            "inner"
        ) \
        .join(
            snapshot(categories_hist, F.col("scu.change_ts")).alias("sc"),
            F.col("s.SupplierCategoryID") == F.col("sc.SupplierCategoryID"),
            "inner"
        ) \
        .join(
            people.alias("p"),
            F.col("s.PrimaryContactPersonID") == F.col("p.PersonID"),
            "inner"
        ) \
        .select(
            F.col("s.SupplierID").alias("WWI Supplier ID"),
            F.col("s.SupplierName").alias("Supplier"),
            F.col("sc.SupplierCategoryName").alias("Category"),
            F.col("p.FullName").alias("Primary Contact"),
            F.col("s.SupplierReference").alias("Supplier Reference"),
            F.col("s.PaymentDays").alias("Payment Days"),
            F.col("s.DeliveryPostalCode").alias("Postal Code"),
            F.col("scu.change_ts").alias("Valid From"),
            F.col("s.ValidTo").alias("Valid To")
        )

    # ---------------------------------------------------------------------------
    # Combine both result sets
    # ---------------------------------------------------------------------------
    combined = cat_change_rows.unionByName(supplier_change_rows)

    # ---------------------------------------------------------------------------
    # Compute the correct [Valid To] based on the next [Valid From] per supplier
    # ---------------------------------------------------------------------------
    w = Window.partitionBy("WWI Supplier ID").orderBy("Valid From")
    combined_with_to = combined.withColumn(
        "next_valid_from",
        F.lead("Valid From").over(w)
    ).withColumn(
        "Valid To",
        F.coalesce(F.col("next_valid_from"), F.lit(END_OF_TIME))
    ).drop("next_valid_from")

    # ---------------------------------------------------------------------------
    # Write the result to the staging table (append mode)
    # ---------------------------------------------------------------------------
    combined_with_to.write.format("delta").mode("append").saveAsTable("Integration.Supplier_Staging")

    # ---------------------------------------------------------------------------
    # End of migration step
    # ---------------------------------------------------------------------------


## Load Transaction Fact

| Field | Value |
| --- | --- |
| Step ID | `load_transaction_fact` |
| Step Type | `FACT_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetTransactionUpdates` |
| Target Objects | `Integration.Transaction_Staging` |


<details>
<summary>AI Migration Contract (click to expand)</summary>

```python
# AI_REQUIRED
#
# ============================================================
# SSIS -> Databricks AI Migration Artifact
# ============================================================
#
# Migration step:
# Load Transaction Fact
#
# Target:
# Databricks PySpark
#
# Migration strategy:
# AI_ASSISTED
#
# Migration patterns:
# ['INCREMENTAL_COMPOSITE']
#
# Complexity signals:
# ['UNION']
#
# ------------------------------------------------------------
# SOURCE PROCEDURE
# ------------------------------------------------------------
#
# Integration.GetTransactionUpdates
#
# The complete source procedure is preserved in artifact
# metadata under:
#
#   source_procedure_definition
#
# The AI migration agent must preserve its semantics.
#
# ------------------------------------------------------------
# DESTINATION PROCEDURE
# ------------------------------------------------------------
#
# Integration.MigrateStagedTransactionData
#
# The complete destination procedure is preserved in
# artifact metadata under:
#
#   destination_procedure_definition
#
# The AI migration agent must analyze its target-side
# semantics before generating the Databricks implementation.
#
# ------------------------------------------------------------
# MIGRATION REQUIREMENT
# ------------------------------------------------------------
#
# Generate executable Databricks PySpark logic equivalent to
# the original SSIS workflow.
#
# The implementation must account for:
#
# 1. Source extraction semantics
# 2. Incremental cutoff handling
# 3. SSIS data-flow mappings
# 4. Staging behavior
# 5. Destination transformation semantics
# 6. Dimension/surrogate-key lookups
# 7. Target fact/dimension loading
# 8. Lineage behavior
# 9. ETL cutoff behavior
# 10. Transaction / consistency semantics
#
# Do not invent business logic that is not supported by the
# supplied source SQL, destination SQL, SSIS metadata, or
# database metadata.
#
# This artifact is an AI input contract, not executable code.

```

</details>


In [0]:
if _step_id in ("all", "load_transaction_fact"):
    # [set_table_name] (CONTROL)
    table_name = "Integration.Transaction_Staging"


    # [get_last_cutoff] (CONTROL)
    last_cutoff = get_last_etl_cutoff(
        table_name=table_name
    )


    # [get_lineage_key] (CONTROL)
    lineage_key = get_lineage_key(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


    # [clear_staging] (STAGING)
    spark.sql("""
    DELETE FROM Integration.Transaction_Staging
    """)


    # [extract_to_staging] (EXTRACT)
    source_df = spark.sql("""
    SELECT CAST(ct.TransactionDate AS date) AS `Date Key`,
               ct.CustomerTransactionID AS `WWI Customer Transaction ID`,
               CAST(NULL AS int) AS `WWI Supplier Transaction ID`,
               ct.InvoiceID AS `WWI Invoice ID`,
               CAST(NULL AS int) AS `WWI Purchase Order ID`,
               CAST(NULL AS nvarchar(20)) AS `Supplier Invoice Number`,
               ct.AmountExcludingTax AS `Total Excluding Tax`,
               ct.TaxAmount AS `Tax Amount`,
               ct.TransactionAmount AS `Total Including Tax`,
               ct.OutstandingBalance AS `Outstanding Balance`,
               ct.IsFinalized AS `Is Finalized`,
               COALESCE(i.CustomerID, ct.CustomerID) AS `WWI Customer ID`,
               ct.CustomerID AS `WWI Bill To Customer ID`,
               CAST(NULL AS int) AS `WWI Supplier ID`,
               ct.TransactionTypeID AS `WWI Transaction Type ID`,
               ct.PaymentMethodID AS `WWI Payment Method ID`,
               ct.LastEditedWhen AS `Last Modified When`
        FROM Sales.CustomerTransactions AS ct
        LEFT OUTER JOIN Sales.Invoices AS i
        ON ct.InvoiceID = i.InvoiceID
        WHERE ct.LastEditedWhen > :LastCutoff
        AND ct.LastEditedWhen <= :NewCutoff

        UNION ALL

        SELECT CAST(st.TransactionDate AS date) AS `Date Key`,
               CAST(NULL AS int) AS `WWI Customer Transaction ID`,
               st.SupplierTransactionID AS `WWI Supplier Transaction ID`,
               CAST(NULL AS int) AS `WWI Invoice ID`,
               st.PurchaseOrderID AS `WWI Purchase Order ID`,
               st.SupplierInvoiceNumber AS `Supplier Invoice Number`,
               st.AmountExcludingTax AS `Total Excluding Tax`,
               st.TaxAmount AS `Tax Amount`,
               st.TransactionAmount AS `Total Including Tax`,
               st.OutstandingBalance AS `Outstanding Balance`,
               st.IsFinalized AS `Is Finalized`,
               CAST(NULL AS int) AS `WWI Customer ID`,
               CAST(NULL AS int) AS `WWI Bill To Customer ID`,
               st.SupplierID AS `WWI Supplier ID`,
               st.TransactionTypeID AS `WWI Transaction Type ID`,
               st.PaymentMethodID AS `WWI Payment Method ID`,
               st.LastEditedWhen AS `Last Modified When`
        FROM Purchasing.SupplierTransactions AS st
        WHERE st.LastEditedWhen > :LastCutoff
        AND st.LastEditedWhen <= :NewCutoff
    """)


    # [resolve_dimension_keys] (LOOKUP_UPDATE)
    dimension_targets = [
        "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type", "Dimension.Payment Method"
    ]

    # Resolve dimension/surrogate keys according to
    # the semantic analysis of the source procedure.

    # Semantic operation:
    # RESOLVE_DIMENSION_KEYS

    # Required target dimensions:
    # "Dimension.Customer", "Dimension.Supplier", "Dimension.Transaction Type", "Dimension.Payment Method"


    # [insert_target] (INSERT)
    # TODO: generate executable INSERT.
    #
    # Target:
    # Fact.Transaction
    #
    # Semantic analysis:
    # {'operation': 'INSERT_TARGET', 'type': 'INSERT', 'description': 'Insert staged records into the target table.', 'target': 'Fact.Transaction', 'features': ['INSERT', 'SELECT_FROM_STAGING']}


    # [complete_lineage] (UPDATE)
    # Mark lineage record as successfully completed.
    complete_lineage(lineage_key)


    # [update_etl_cutoff] (UPDATE)
    # Advance ETL cutoff after successful load.
    update_etl_cutoff(
        table_name=table_name,
        cutoff_time=new_cutoff
    )


## Load Transaction Type Dimension

| Field | Value |
| --- | --- |
| Step ID | `load_transaction_type_dimension` |
| Step Type | `DIMENSION_LOAD` |
| Strategy | `AI_ASSISTED` |
| Dependencies | None |
| Source Objects | `Integration.GetTransactionTypeUpdates` |
| Target Objects | `Integration.TransactionType_Staging` |


**AI-Generated Code** (status: `GENERATED`)

**Assumptions:**
- Source tables Application.TransactionTypes and Application.TransactionTypes_Archive are Delta tables with columns TransactionTypeID, TransactionTypeName, ValidFrom, ValidTo.
- Both source tables contain row‑level validity columns (ValidFrom, ValidTo) as described in temporal_context.
- The staging table Integration.TransactionType_Staging exists and is a Delta table compatible with the column names produced by the extraction step.
- Dimension table Dimension.`Transaction Type` has the columns WWI Transaction Type ID, Transaction Type, Valid From, Valid To, Lineage Key and is a Delta table.
- Integration.Lineage and Integration.`ETL Cutoff` tables contain the columns referenced in the destination procedure (Lineage Key, Table Name, Data Load Completed, Was Successful, Source System Cutoff Time, Cutoff Time).
- The constant END_OF_TIME ('9999-12-31 23:59:59.9999999') fits the timestamp type of the target columns.
- No concurrent processes are modifying the same dimension or staging tables during execution.


In [0]:
if _step_id in ("all", "load_transaction_type_dimension"):
    from pyspark.sql import SparkSession, Window
    import pyspark.sql.functions as F

    spark = SparkSession.builder.getOrCreate()

    END_OF_TIME = "9999-12-31 23:59:59.9999999"

    def load_transaction_type_dimension(last_cutoff: str, new_cutoff: str):
        """Load Transaction Type dimension for the given cutoff window.

        Parameters
        ----------
        last_cutoff: str
            Lower bound (exclusive) of the validity window (ISO‑8601 timestamp string).
        new_cutoff: str
            Upper bound (inclusive) of the validity window (ISO‑8601 timestamp string).
        """
        # ---------------------------------------------------------------------
        # 1. Extract changed transaction‑type rows (source procedure logic)
        # ---------------------------------------------------------------------
        df_current = spark.table("Application.TransactionTypes")
        df_archive = spark.table("Application.TransactionTypes_Archive")
        df_all = df_current.unionByName(df_archive)

        # Identify the change timestamps for each TransactionTypeID
        df_changes = (
            df_all
            .filter((F.col("ValidFrom") > F.lit(last_cutoff)) & (F.col("ValidFrom") <= F.lit(new_cutoff)))
            .select("TransactionTypeID", "ValidFrom")
            .distinct()
        )

        # For each (TransactionTypeID, ValidFrom) retrieve the row that was valid at that moment
        df_as_of = (
            df_changes.alias("c")
            .join(
                df_all.alias("t"),
                (F.col("c.TransactionTypeID") == F.col("t.TransactionTypeID"))
                & (F.col("c.ValidFrom") >= F.col("t.ValidFrom"))
                & (F.col("c.ValidFrom") < F.col("t.ValidTo")),
                "inner",
            )
            .select(
                F.col("t.TransactionTypeID").alias("WWI Transaction Type ID"),
                F.col("t.TransactionTypeName").alias("Transaction Type"),
                F.col("c.ValidFrom").alias("Valid From"),
                F.col("t.ValidTo").alias("TempValidTo"),
            )
        )

        # Compute the final Valid To by looking at the next Valid From for the same ID
        w = Window.partitionBy("WWI Transaction Type ID").orderBy("Valid From")
        df_final = (
            df_as_of
            .withColumn("next_valid_from", F.lead("Valid From").over(w))
            .withColumn("Valid To", F.coalesce(F.col("next_valid_from"), F.lit(END_OF_TIME)))
            .drop("TempValidTo", "next_valid_from")
        )

        # Persist the result to the staging table
        df_final.write.format("delta").mode("append").saveAsTable("Integration.TransactionType_Staging")

        # ---------------------------------------------------------------------
        # 2. Load staged data into the dimension (destination procedure logic)
        # ---------------------------------------------------------------------
        # Retrieve the pending lineage key
        lineage_rows = spark.sql(
            """
            SELECT `Lineage Key`
            FROM Integration.Lineage
            WHERE `Table Name` = 'transaction_type' AND `Data Load Completed` IS NULL
            ORDER BY `Lineage Key` DESC
            LIMIT 1
            """
        ).collect()
        if not lineage_rows:
            raise RuntimeError("No pending lineage record found for Transaction Type dimension.")
        lineage_key = lineage_rows[0][0]

        # Determine the earliest Valid From per WWI Transaction Type ID in the staging area
        rows_to_close = spark.sql(
            """
            SELECT `WWI Transaction Type ID`, MIN(`Valid From`) AS `Valid From`
            FROM Integration.TransactionType_Staging
            GROUP BY `WWI Transaction Type ID`
            """
        )
        rows_to_close.createOrReplaceTempView("RowsToCloseOff")

        # Close off existing open‑ended rows in the dimension
        spark.sql(
            f"""
            UPDATE Dimension.transaction_type AS dim
            SET dim.`Valid To` = rco.`Valid From`
            FROM RowsToCloseOff rco
            WHERE dim.`WWI Transaction Type ID` = rco.`WWI Transaction Type ID`
              AND dim.`Valid To` = '{END_OF_TIME}'
            """
        )

        # Insert the new rows from staging, attaching the lineage key
        spark.sql(
            f"""
            INSERT INTO Dimension.transaction_type
                (`WWI Transaction Type ID`, `Transaction Type`, `Valid From`, `Valid To`, `Lineage Key`)
            SELECT `WWI Transaction Type ID`, `Transaction Type`, `Valid From`, `Valid To`, {lineage_key}
            FROM Integration.TransactionType_Staging
            """
        )

        # Mark the lineage record as completed successfully
        spark.sql(
            f"""
            UPDATE Integration.Lineage
            SET `Data Load Completed` = current_timestamp(),
                `Was Successful` = 1
            WHERE `Lineage Key` = {lineage_key}
            """
        )

        # Advance the ETL cutoff for this source table
        spark.sql(
            f"""
            UPDATE Integration.`ETL Cutoff`
            SET `Cutoff Time` = (
                SELECT `Source System Cutoff Time`
                FROM Integration.Lineage
                WHERE `Lineage Key` = {lineage_key}
            )
            WHERE `Table Name` = 'transaction_type'
            """
        )

        # Optional: clear staging if the workflow expects it (not part of original proc)
        # spark.sql("TRUNCATE TABLE Integration.TransactionType_Staging")

        return {
            "lineage_key": lineage_key,
            "rows_inserted": df_final.count()
        }
